In [2]:
!pip install -U ipywidgets

In [3]:
import ipywidgets as widgets
from IPython.display import display
import os
from datetime import datetime

In [4]:
save_folder = "Desktop"
os.makedirs(save_folder, exist_ok=True)

print("Images will be saved in:", os.path.abspath(save_folder))

Images will be saved in: /Users/hotboytraidat/Desktop/project data/Desktop


In [5]:
upload = widgets.FileUpload(
    accept='.tif,.tiff,.png,.jpg,.jpeg',
    multiple=True,
    description="Upload images"
)

marker = widgets.Text(description="Marker")

dilution = widgets.Dropdown(
    options=["1:50","1:100","1:200","1:300","1:500","1:800","1:1000","1:2000","Other"],
    description="Dilution"
)

dilution_other = widgets.Text(description="Other")

buffer = widgets.Dropdown(
    options=["Citrate","HTTR","EDTA","Other"],
    description="Buffer"
)

buffer_other = widgets.Text(description="Other")

magnification = widgets.Dropdown(
    options=["10X","20X","40X"],
    description="Magnification")

user = widgets.Text(description="User")

rename_button = widgets.Button(description="Rename Images")

output = widgets.Output()

def rename_images(b):
    with output:
        output.clear_output()

        if not upload.value:
            print("Upload images first")
            return

        m = marker.value.strip()

        d = dilution.value
        if d == "Other":
            d = dilution_other.value.strip()

        buf = buffer.value
        if buf == "Other":
            buf = buffer_other.value.strip()

        mag = magnification.value
        u = user.value.strip().upper()

        date = datetime.now().strftime("%Y%m%d")

        if isinstance(upload.value, dict):
            files = list(upload.value.values())
            get_name = lambda f: f["metadata"]["name"] if "metadata" in f else f["name"]
            get_content = lambda f: f["content"]
        else:
            files = list(upload.value)
            get_name = lambda f: f["name"]
            get_content = lambda f: f["content"]

        for i, fileinfo in enumerate(files, start=1):
            name = get_name(fileinfo)
            ext = os.path.splitext(name)[1]
            d_clean = d.replace(":", "-")

            newname = f"{m}_{d_clean}_{buf}_{mag}_{u}_{date}_ROI{i:03d}{ext}"
            path = os.path.join(save_folder, newname)

            with open(path, "wb") as f:
                f.write(get_content(fileinfo))

            print("Saved:", newname)

        print("\nSaved folder:", os.path.abspath(save_folder))

rename_button.on_click(rename_images)

display(
    widgets.VBox([
        upload,
        marker,
        dilution,
        dilution_other,
        buffer,
        buffer_other,
        magnification,
        user,
        rename_button,
        output
    ])
)